In [4]:
import asyncio
from datetime import datetime, timedelta
from uuid import uuid4
from src.backend.db.dbFacade import DBFacade
from src.backend.models.db_models import *

db = DBFacade()
#await db.drop_all_tables()
await db.create_tables()

[2025-10-07 13:03:20] [INFO] Tables created successfully!


In [ ]:
client_id = await db.get_client_id_by_meet_id('7b6ef059-e147-4dc4-bb58-159fd2d57aa7')

In [ ]:
print(client_id)

In [ ]:
c = await db.get_client_full_info(client_id)

In [ ]:
print(c)
p = c["persons"][0]
name = " ".join(x for x in [p.first_name, getattr(p, "middle_name", "") or "", p.last_name] if x).strip()
print(name)

In [ ]:
p = c.persons[0]
name = " ".join(x for x in [p.first_name, getattr(p, "middle_name", ""), p.last_name] if x).strip()
print(name)

In [ ]:
await db.delete_table('meet')

In [1]:
import tiktoken
from src.backend.core.baseFacade import BaseFacade
from src.backend.prompts.promptFacade import PromptFacade
from src.backend.models.llm_models import NotesResponse, SummarizerResponse, OverviewResponse, ActionItemsResponse
from src.backend.db.dbFacade import DBFacade
from src.backend.modules.adviser_pipeline import AdvisorProcessor
from src.backend.modules.transcriber import Transcriber
from src.backend.core.baseFacade import BaseFacade
from src.backend.db.dbFacade import DBFacade
from src.backend.models.db_models import MeetUpdate
from src.backend.modules.meetingAnalizer import MeetingAnalizer
from src.backend.llm.routerFacade import RouterAgent
from src.backend.vector_db.qdrant_Facade import VectorDBFacade
from src.backend.vector_db.sql_to_vector import SQLQdrantSynchronizer
db = DBFacade()
from src.backend.utils.logger import CustomLog  
meeting_analizer = MeetingAnalizer()
full_transcript_text = ''' 
Michael Chen: Good afternoon, this is Michelle Turner with Summit Wealth Advisors. Am I speaking with David Brooks?David Brooks: Yes, this is David.Michael Chen: Hi David, how are you doing today?David Brooks: Doing alright, thanks. Just finishing up a long morning at the office.Michael Chen: I hear you. I’ll keep this brief. The reason for my call is that we’re reviewing retirement strategies for clients who already have 401(k) plans, like yours through Lone Star Energy. I’d like to walk you through a few options that might complement your existing savings and protection plan. Would you have a few minutes?David Brooks: Sure, I can give you about 15 minutes.Michael Chen: Perfect. First, I see you’ve been contributing regularly to your 401(k). That’s excellent. One of the things we recommend is adding a supplemental Roth IRA. This allows you to diversify tax treatment—so part of your retirement income is pre-tax and part is tax-free. Have you considered opening a Roth outside of your employer plan?David Brooks: I’ve heard about it, but honestly, I’m already putting 3% into the Roth portion of my 401(k). I don’t know if I want to manage another account right now.Michael Chen: Totally understandable. It can feel like a lot of moving pieces.Michael Chen: Another option we see clients like you benefit from is a 529 college savings plan for your kids. You’ve got Sarah and Ethan, correct?David Brooks: Yeah, that’s right. Sarah’s in high school and Ethan’s starting freshman year next year.Michael Chen: Great ages to start thinking about college. The 529 allows you to invest funds that grow tax-free as long as they’re used for education. Texas doesn’t have state income tax, but federal tax advantages still apply. Does that sound like something you’d be interested in?David Brooks: That actually makes sense. College costs are on my mind lately. I’d like to know more about setting that up.Michael Chen: Excellent, I’ll make a note to prepare some details for you.Michael Chen: Next, a lot of clients in your income range look at life insurance with a cash value component. This serves two purposes: protection for your family and a tax-advantaged savings vehicle. Do you already have any life insurance outside of your employer benefits?David Brooks: Just what’s through Lone Star. I think it’s a basic term policy. Honestly, I don’t like the idea of mixing investments with insurance. I’d prefer to keep them separate.Michael Chen: That’s a fair perspective. Not everyone is comfortable with whole life policies.Michael Chen: Another product we sometimes recommend is a low-risk bond fund. With the market being volatile, it can help balance your equity-heavy 401(k). You wouldn’t need to commit much, maybe $10–15K, just to add a stable layer to your portfolio.David Brooks: Hm. I’m not sure about that right now. My focus is more on growth, and bonds just don’t excite me. I’d rather keep putting extra into my 401(k) or the 529 you mentioned.Michael Chen: Understood. Growth focus makes sense at your age.Michael Chen: Lastly, we also offer long-term care insurance. It’s something many clients overlook until later in life, but locking it in now keeps premiums manageable. Have you thought about that?David Brooks: I’ve read about it, but honestly, I don’t think I’m ready to add that expense. I’d rather focus on college and retirement right now. Maybe revisit it in 10 years.Michael Chen: That’s a reasonable plan.Michael Chen: So to recap, you’re not interested in an outside Roth IRA or whole life insurance at this time. You’re also passing on the bond fund and long-term care for now. But you are interested in moving forward with a 529 plan for Sarah and Ethan. Did I capture that correctly?David Brooks: Exactly. The 529 is the one I want to explore further.Michael Chen: Perfect. I’ll prepare a personalized 529 proposal and send it over to your email: david.brooks@email.com. Then we can set up a follow-up call to go over the numbers. Does Thursday afternoon work?David Brooks: Thursday works. Thanks, Michelle.Michael Chen: Great, I’ll send you a calendar invite. Have a good rest of your day, David.David Brooks: You too. '''


In [2]:
overview = await meeting_analizer.generate_overview(full_transcript_text)

In [3]:
print(overview)

overview=['Mary discussed her financial situation and mentioned that by the next meeting she will have a better sense of how the $50 monthly contribution fits in her budget, and may be able to switch Emma to after-school care instead of the more expensive daycare.', 'Michael Chen agreed to send Mary the life insurance application and set up the 529 account. He will also email her a summary of their conversation and educational materials about both products.', "Michael confirmed Mary's email address for communication.", "Mary expressed appreciation for Michael's time and said this feels like a great starting point for her financial planning.", "Michael encouraged Mary, affirming that she is taking important steps for her and Emma's future, and offered continued support for any questions she may have."] title='Financial Planning Meeting: Life Insurance and 529 Setup'


In [2]:
s = await meeting_analizer.summarize(full_transcript_text,'520e646f-83e5-4e1f-b6b3-b7f6c7d0815a')

[2025-10-07 11:58:45] [INFO] Strategy analysis completed
[2025-10-07 11:58:52] [INFO] Meeting analysis completed
[2025-10-07 11:58:53] [INFO] Generated 10 tags
[2025-10-07 11:58:53] [INFO] Analysis completed successfully


In [3]:
print(s)

summary='# COMPREHENSIVE FINANCIAL ADVISOR REPORT\n\n## MEETING ANALYSIS\n# MEETING ANALYSIS\n\n**Products Discussed:**\n- Roth Ira: negative - low interest [“I don’t know if I want to manage another account right now.”]\n- 529 College Savings Plan: positive - high interest [“That actually makes sense. College costs are on my mind lately. I’d like to know more about setting that up.”]\n- Life Insurance Whole: negative - low interest [“I don’t like the idea of mixing investments with insurance. I’d prefer to keep them separate.”]\n- Long Term Care Ltc: negative - low interest [“I don’t think I’m ready to add that expense. I’d rather focus on college and retirement right now.”]\n\n**Decisions Made:**\n- Accept: 529 College Savings Plan - “The 529 is the one I want to explore further.”\n- Reject: Roth Ira - “I don’t know if I want to manage another account right now.”\n- Reject: Life Insurance Whole - “I don’t like the idea of mixing investments with insurance.”\n- Reject: Long Term Care 

In [6]:
d = await meeting_analizer.generate_action_items(full_transcript_text,'7b6ef059-e147-4dc4-bb58-159fd2d57aa7')
print(d)

Life Insurance Whole,529 College Savings Plan


In [ ]:
meets = await db.get_all_meets()

In [ ]:
meets

In [ ]:
meet_messages = await db.get_chat_messages_by_meet_id("d1616201-18da-492c-9545-411d6192b2bc")

In [ ]:
meet_messages

In [ ]:
new_meet = await db.update_meet("055cfa48-f0c2-41af-a84b-5c109b15d5ee", MeetUpdate(transcript="New full transcript", summary="New summary"))

In [ ]:
last_meet = await db.get_meet_by_id("f427183c-8358-49a0-995a-64956ba0be63")


In [ ]:
last_meet.model_dump().get("transcript")

In [ ]:
new_meet

In [ ]:
users = await db.get_all_users()

In [ ]:
users

In [ ]:
companies = await db.get_all_companies()

In [ ]:
companies

In [ ]:
company = await db.create_company(CompanyCreate(
    title="SuperCorp",
    email_domen="supercorp.com",
    subscription="premium",
    subscription_term=datetime.utcnow() + timedelta(days=365),
    registration_date=datetime.utcnow()
))

In [ ]:
company

In [ ]:
companies = await db.get_all_companies()

In [ ]:
companies

In [ ]:
user = await db.create_user(UserCreate(
    email="john@supercorp.com",
    company_id=company.id,
    username="John",
    password="pass123",
    role="manager",
    gender="male",
    language="en"
))

In [ ]:
user

In [ ]:
user2 = await db.create_user(UserCreate(
    email="solo@example.com",
    company_id="7bfef7ec-0e70-482f-adbb-bce9bea7f9de",
    username="Anna",
    password="secret",
    role="freelancer",
    gender="female",
    language="en"
))

In [ ]:
user2

In [ ]:
meet1 = await db.create_meet(MeetCreate(
    user_id=user.id,
    title="Weekly Standup",
    summary="Discuss project updates",
    date=datetime.now(),
    duration=60,
    overview="Overview text",
    notes="Some notes",
    action_items="Do tasks",
    transcript="Transcript text",
    language="en",
    tags="team,weekly"
))

In [ ]:
meet1

In [ ]:
meet2 = await db.create_meet(MeetCreate(
    user_id=user2.id,
    title="Client Call",
    date=datetime.now(),
    language="en"
))

In [ ]:
meet2

In [ ]:
messages = await db.get_meeting_chat_message_by_id("38a6f98d-21cb-4f1e-8060-ae2455a0e063'")

In [ ]:
messages

In [ ]:
message = await db.create_meeting_message(MeetingMessageCreate(
    meet_id=meet1.id,
    time=datetime.utcnow(),
    email=user.email,
    content="Hello team!"
))

In [ ]:
message

In [ ]:
chat_msg = await db.create_meeting_chat_message(MeetingChatMessageCreate(
    meet_id=meet1.id,
    time=datetime.utcnow(),
    role="manager",
    content="What is the summary of the previous meeting"
))

In [ ]:
msgs = await db.get_chat_messages_by_meet_id(meet1.id)

In [ ]:
msgs

In [ ]:
chat_msg

In [ ]:
participant = await db.create_participant(ParticipantCreate(
    meet_id=meet1.id,
    time=datetime.utcnow(),
    email="guest@supercorp.com"
))

In [ ]:
participant

In [ ]:
updated_meet = await db.update_meet(meet1.id, MeetUpdate(title="Weekly Sync Updated"))

In [ ]:
updated_meet

In [ ]:
meets = await db.get_all_meets()

In [ ]:
meets

In [ ]:
user2_meet = await db.get_meets_by_user_id(user2.id)

In [ ]:
user2_meet